# Dependencies

In [ ]:
# pip install "unstructured[all-docs]"
# %pip install -Uq langchain_chroma 
# %pip install -Uq langchain langchain_community
# %pip install -Uq python_dotenv
# %pip install -Uq langchain_ollama langchain_huggingface

In [29]:
import json 
from typing import List

# Unstructured for Document Parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# Langchain Components
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama



In [2]:
def partition_document(file_path: str):
    """Extract elements from pdf using Unstructured"""
    print(f"Partitioning Document {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res", # use the most accurate (but slower) processing method for extraction
        infer_table_structure=True, # Keep Tables as structured HTML not jumbled text
        extract_image_block_types=["Image"], # Grab Images found in the PDF
        extract_image_block_to_payload=True # Store Image as base64 data you can actually use
    )

    print(f"Extracted {len(elements)} elements")
    return elements


In [3]:
file_path = "./NIPS-2017-attention-is-all-you-need-Paper.pdf"
elements = partition_document(file_path)

Partitioning Document ./NIPS-2017-attention-is-all-you-need-Paper.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 14656.32it/s]


Extracted 173 elements


In [13]:
elements[34].to_dict()

{'type': 'Image',
 'element_id': '4b987fa3f00da690d6a7759ad8bfb8d6',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Add & Norm Feed Forward Nx | (Gada Nom) Add & Norm Masked Multi-Head Multi-Head Attention Attention Positional Encoding O° Positional 4 oe Encoding Input Output Embedding Embedding Inputs Outputs (shifted right)',
 'metadata': {'coordinates': {'points': ((np.float64(545.9972222222221),
     np.float64(200.00555555555542)),
    (np.float64(545.9972222222221), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(200.00555555555542))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-25T22:24:15',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': '.',
  'filename': 

In [7]:
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

### Gather All Images

In [14]:
images = [element for element in elements if element.category == 'Image']
print(f"Found: {len(images)} Images")

Found: 3 Images


In [16]:
images[2].to_dict()

{'type': 'Image',
 'element_id': 'b33ef76f8c444cc3096e8d7dedb00090',
 'text': 'Linear',
 'metadata': {'coordinates': {'points': ((np.float64(963.2555555555555),
     np.float64(229.68611111111082)),
    (np.float64(963.2555555555555), np.float64(742.486111111111)),
    (np.float64(1297.2555555555555), np.float64(742.486111111111)),
    (np.float64(1297.2555555555555), np.float64(229.68611111111082))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-25T22:24:15',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 4,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': '.',
  'filename': 'NIPS-2017-attention-is-all-you-need-Paper.pdf'}}

### Gather All Tables

In [17]:
tables = [element for element in elements if element.category == 'Table']
print(f"Found: {len(tables)} Tables")

Found: 3 Tables


In [19]:
tables[0].to_dict()

{'type': 'Table',
 'element_id': '25b9167c201f0fe4485e4890224185e5',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9252336025238037,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(324.720703125),
     np.float64(314.848876953125)),
    (np.float64(324.720703125), np.float64(518.4424438476562)),
    (np.float64(1359.559326171875), np.float64(518.4424438476562)),
    (np.float64(1359.559326171875), np.float64(314.848876953125))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-04-25T22:24:15',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention

## Create Chunks from Elements Based on Title

In [20]:
def create_chunks_by_title(elements, max_chars = 3000, new_chunk_start_after = 2400, merge_chunks_under_chars = 500 ):
    """Creating Intelligent chunks using title based strategy"""
    print("Creating smart chunks...")

    chunks = chunk_by_title(
        elements, # The Parsed PDF elements from previous step
        max_characters=max_chars, # Hard Limit - Never exceeds 3000 character per chunk
        new_after_n_chars=new_chunk_start_after, # Try to start new Chunk after N chars (2400)
        combine_text_under_n_chars=merge_chunks_under_chars # Merge tiny chunks under 500 chars
    )

    print(f"Created: {len(chunks)} Chunks")

    return chunks


In [22]:
# Create Chunks
chunks = create_chunks_by_title(elements)

Creating smart chunks...
Created: 21 Chunks


In [23]:
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [24]:
chunks[2].to_dict()

{'type': 'CompositeElement',
 'element_id': '8ca2646b-9491-403b-8bf1-c89563c56432',
 'text': 'In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.\n\n2 Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural networks as basic building block, computing hidden representations in parallel for all input and output positions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This make

In [35]:
# View Original Element
chunks[4].metadata.orig_elements
# chunks[4].metadata.orig_elements[3].to_dict()

In [42]:
def seperate_content_types(chunk):
    """Analyze what type of content are in a Chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle Tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            # Handle Images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
    return content_data


In [31]:
def create_AI_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI Enhanced Summary for Mixed Content"""

    try:

        llm_model = ChatOllama(
            model="deepseek-v3.1:671b-cloud",
            temperature=0
        )

        prompt_text = f"""You are creating a searchable description for document content retrieval.
        
        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}
        """

        # Add tables if Present

        if tables:
            prompt_text += "TABLES:\n"

            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}: \n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers and data points from text and tables
                2. Main topics & concepts discussed
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:
                """

        # Build message content starting with text
        message_content = [{'type': 'text', 'text': prompt_text}]

        # Add Images to the message
        if images:
            for image in images:
                message_content.append({
                    'type': 'image_url',
                    'image_url': {'url': f"data:image/jpeg;base64,{image}"}
                })
        
        message = HumanMessage(content=message_content)

        response = llm_model.invoke(message)
        return response.content
    
    except Exception as ex:
        print(f" AI Summary Failed: {ex}")


        # Fallback to Simple Summary
        summary = f"{text[:300]}..."

        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Containes {len(images)} image(s)]"

        return summary
    




In [47]:
def summarize_chunks(chunks):
    """Process All Chunks with AI Summaries"""
    print("Processing Chunks with AI Summaries")

    langchain_documents = []

    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f">>  Processing Chunk  {current_chunk}/{total_chunks} -----------")

        # Analyze Chunk Content
        content_data = seperate_content_types(chunk)

        # Debug Prints
        print(f"------->>  Types Found: {content_data['types']} ------------")
        print(f"------->>  Tables: {len(content_data['tables'])} Tables ------------")
        print(f"------->>  Images: {len(content_data['images'])} Images ------------")

        # Create AI-Enhanced Summary if Chunk has Tables & Images
        if content_data['tables'] or content_data['images']:
            print(f"-------->> Creating AI summary for Mixed Content...")

            try:
                enhanced_content = create_AI_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images'],
                )

                print(f"------->> AI Summary Created Successfully")
                print(f"------->> Enhanced Content Preview: {enhanced_content[:200]}...")

            except Exception as e:
                print(f"AI Summary Failed...")
                enhanced_content = content_data['text']

        # If chunk does not have any Table or Image
        else:
            print(f"------->> Using only raw Text (No Tables/Images)")
            enhanced_content = content_data['text']

        
        # Create LangChain Document with Rich Metadata
        doc = Document(
            page_content=enhanced_content,
            metadata = {
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )

        langchain_documents.append(doc)

    print(f">> Processed {len(langchain_documents)} Chunks")
    return langchain_documents
    

In [48]:
# Process Chunks with AI
processed_chunks = summarize_chunks(chunks)

Processing Chunks with AI Summaries
>>  Processing Chunk  1/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  2/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  3/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  4/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  5/21 -----------
------->>  Types Found: ['text', 'image'] ------------
------

In [46]:
processed_chunks[1].to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'document', 'Document'],
 'kwargs': {'metadata': {'original_contet': '{"raw_text": "1 Introduction\\n\\nRecurrent neural networks, long short-term memory [12] and gated recurrent [7] neural networks in particular, have been \\ufb01rmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [29, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [31, 21, 13].\\n\\n\\u2217Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started\\n\\nthe effort to evaluate this idea. Ashish, with Illia, designed and implemented the \\ufb01rst Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head attention and the parameter-free position representatio

In [49]:
def export_chunks_to_json(chunks, file_name="export_chunks.json"):
    """Export Processed chunks to clean JSON Format"""
    export_data = []

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }

        export_data.append(chunk_data)

    # Save to File
    with open(file_name, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f" Exported {len(export_data)} chunks to {file_name}")

    return export_data

In [50]:
json_data = export_chunks_to_json(processed_chunks)

 Exported 21 chunks to export_chunks.json


In [51]:
json_data

[{'chunk_id': 1,
  'enhanced_content': 'Attention Is All You Need\n\nAshish Vaswani∗ Google Brain avaswani@google.com\n\nNoam Shazeer∗ Google Brain noam@google.com\n\nNiki Parmar∗ Google Research nikip@google.com\n\nJakob Uszkoreit∗ Google Research usz@google.com\n\nLlion Jones∗\n\nGoogle Research llion@google.com\n\nAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu\n\nŁukasz Kaiser∗ Google Brain lukaszkaiser@google.com\n\nIllia Polosukhin∗ ‡\n\nillia.polosukhin@gmail.com\n\nAbstract\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being 

In [52]:
def create_vector_store(documents, persist_directory="./../db/chroma_db"):
    """Create & Persist Chroma DB Vector Store"""
    print("Creating Embeddings and Storing in Chroma DB...")

    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    print("---------Creating ChromaDB Vector Store -----------")
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )

    print("-------- Finished Creating Vector Store ----------")

    print(f"Vector Store Created & Saved to - {persist_directory}")
    
    return vector_store


In [54]:
db = create_vector_store(processed_chunks)

Creating Embeddings and Storing in Chroma DB...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3586.45it/s]


---------Creating ChromaDB Vector Store -----------
-------- Finished Creating Vector Store ----------
Vector Store Created & Saved to - ./../db/chroma_db


## Complete Ingestion Pipeline

In [59]:
def final_Ingestion_pipeline(pdf_path: str):
    """Run the complete RAG Ingestion Pipeline"""
    print("=" * 50)
    print("Starting RAG Ingestion Pipeline")

    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)

    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)

    # Step 3: AI Summarization
    summarized_chunks = summarize_chunks(chunks)

    # Step 4: Vector Store
    db = create_vector_store(summarized_chunks, "../db1/chroma_db")

    print("=" * 50)
    print("Pipeline Completed Successfully")

    print("=" * 50)
    return db




In [60]:
file_path = "./NIPS-2017-attention-is-all-you-need-Paper.pdf"
db = final_Ingestion_pipeline(file_path)

Starting RAG Ingestion Pipeline
Partitioning Document ./NIPS-2017-attention-is-all-you-need-Paper.pdf
Extracted 173 elements
Creating smart chunks...
Created: 21 Chunks
Processing Chunks with AI Summaries
>>  Processing Chunk  1/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  2/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  3/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 Images ------------
------->> Using only raw Text (No Tables/Images)
>>  Processing Chunk  4/21 -----------
------->>  Types Found: ['text'] ------------
------->>  Tables: 0 Tables ------------
------->>  Images: 0 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17943.73it/s]


---------Creating ChromaDB Vector Store -----------
-------- Finished Creating Vector Store ----------
Vector Store Created & Saved to - ../db1/chroma_db
Pipeline Completed Successfully


### Query

In [66]:
def generate_answer(rt_chunks, query):
    """Generated Final Answer using Multi-Model Content"""

    try:
        llm_Model = ChatOllama(
            model="deepseek-v3.1:671b-cloud",
            temperature=0
        )

        prompt_txt = f"""Based on the following documents, Please answer this question: {query}
        CONTENT TO ANALYZE:
        """

        for i, chunk in enumerate(rt_chunks):
            prompt_txt += f"---Document {i+1} ----\n"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])

                # Add Raw TEXT
                raw_txt = original_data.get("raw_text", "")
                if raw_txt:
                    prompt_txt += f"TEXT: \n{raw_txt}\n\n"
                
                # Add Tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_txt += 'TABLES:\n'
                    for j, table in enumerate(tables_html):
                        prompt_txt += f"Table {j+1}:\n{table}\n\n"
                
            prompt_txt += "\n"
        
        prompt_txt += """
        Please provide a clear, comprehensive answer using the text, tables and images above. If the document don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."
        
        ANSWER:"""


        message_content = [{"type": "text", "text": prompt_txt}]

        # Add All Images from all chunks
        for chunk in rt_chunks:
            if 'original_content' in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])

                images_base64 = original_data.get('images_base64', [])

                for image in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64, {image}"}
                    })

        message = HumanMessage(content=message_content)
        response = llm_Model.invoke([message])

        return response.content
    
    except Exception as e:
        print("=" * 50)
        print(f"Answer Generation Failed {e}")
        # print()
        print("=" * 50)
        return "Sorry, I encountered an error while generating the answer" 
    

In [67]:
# Query
query = "What are the two main components of Transformer Architecture?"

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})
retrieved_chunks = retriever.invoke(query)


In [ ]:
# Method 2: Similarity with Score Threshold

print("\n====>> Method 2: Similarity with Score Threshold ====")

retriever = db.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.3 # Only return docs with similarity >= 0.3
    }
)
retrieved_chunks = retriever.invoke(query)

In [ ]:
# Method 3: Maximum Marginal Relevance (MMR) ---

print("\n====>> Method 3: Maximum Marginal Relevance (MMR) ====")

retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k": 3, # final no. of docs
        "fetch_k": 10, # Initial pool to select from 
        "lambda_mult": 0.5 # 0 - Diversity, 1 - for relevance

    }
)

In [69]:
final_answer = generate_answer(retrieved_chunks, query)
print(final_answer)

Based on the provided documents, the two main components of the Transformer Architecture are the **Encoder** and the **Decoder**.

This is explicitly stated in Document 2, Section 3: "Most competitive neural sequence transduction models have an encoder-decoder structure... The Transformer follows this overall architecture... for both the encoder and decoder."

Document 3 further supports this by describing the Transformer as a model that maps an input sequence to a sequence of continuous representations (the encoder's function) and then generates an output sequence (the decoder's function).


In [58]:
export_result = export_chunks_to_json(retrieved_chunks, "rag_results.json")

 Exported 3 chunks to rag_results.json
